# 00 · Overview & setup — CFTR Variant Toolkit

A beginner-friendly, **honest** tour of the computational tools used to interpret
CFTR variants. Each notebook takes one tool (or tool family), explains *what it
is*, *what its score means*, *the decision threshold and why*, and *how to get the
real data* — then notebook 07 combines them into the A1/A2 worklists and notebook
08 covers the methodology pitfalls.

This notebook just checks your setup and shows the honest data-provenance map.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
print("Python :", sys.version.split()[0])
print("toolkit:", pathlib.Path(tk.__file__).name)
print("cache  :", tk.CACHE_DIR, "->", "FOUND" if tk.CACHE_DIR.exists() else "MISSING")

Python : 3.13.14
toolkit: toolkit.py
cache  : <project>\_tmp_fetch -> FOUND


## The one thing to understand before anything else: REAL vs DEMO

This toolkit is deliberately transparent about what is a *real* download/prediction
and what is a small *hand-curated demo table* shipped so the notebooks run
anywhere. **Never quote a DEMO number as a finding.**

In [2]:
prov = pd.DataFrame([
  ("gnomAD v4 (allele freq)", "REAL",  "~2,466 missense / ~1,085 non-coding", "01"),
  ("AlphaMissense",           "REAL",  "genome-wide (all CFTR missense)",     "02"),
  ("EVE",                     "DEMO",  "~13 curated variants",                "03"),
  ("ESM1b",                   "DEMO",  "~13 curated variants",                "03"),
  ("REVEL",                   "DEMO",  "~13 curated variants",                "04"),
  ("PrimateAI",               "DEMO",  "~13 curated variants",                "04"),
  ("ClinVar",                 "REAL",  "genome-wide",                         "05"),
  ("CFTR2 class",             "DEMO",  "~13 curated variants",                "05"),
  ("SpliceAI",                "DEMO",  "9 curated splice variants",           "06"),
  ("Pangolin",                "DEMO",  "9 curated splice variants",           "06"),
  ("CADD",                    "REAL",  "live per-variant API",                "06"),
], columns=["source","status","coverage","notebook"])
prov

,source,status,coverage,notebook
0,gnomAD v4 (allele freq),REAL,"~2,466 missense / ~1,085 non-coding",01
1,AlphaMissense,REAL,genome-wide (all CFTR missense),02
2,EVE,DEMO,~13 curated variants,03
3,ESM1b,DEMO,~13 curated variants,03
4,REVEL,DEMO,~13 curated variants,04
5,PrimateAI,DEMO,~13 curated variants,04
6,ClinVar,REAL,genome-wide,05
7,CFTR2 class,DEMO,~13 curated variants,05
8,SpliceAI,DEMO,9 curated splice variants,06
9,Pangolin,DEMO,9 curated splice variants,06


In [3]:
# quick proof the REAL loaders work against the cache
print("gnomAD missense :", len(tk.load_gnomad_missense()), "(REAL)")
print("AlphaMissense   :", len(tk.load_alphamissense()), "(REAL, genome-wide)")
print("ClinVar         :", len(tk.load_clinvar()), "(REAL)")
print("EVE demo        :", len(tk.load_eve()), "(DEMO)")
print("splice demo     :", len(tk.load_splice_demo()), "(DEMO)")

gnomAD missense : 2466 (REAL)
AlphaMissense   : 8597 (REAL, genome-wide)
ClinVar         : 2801 (REAL)
EVE demo        : 13 (DEMO)
splice demo     : 9 (DEMO)


## How to read this toolkit

| Notebook | Tool(s) | Real? |
|---|---|---|
| **01** | gnomAD — population allele frequency | REAL |
| **02** | AlphaMissense — the one real genome-wide predictor | REAL |
| **03** | EVE, ESM1b — unsupervised (evolution / protein LM) | demo |
| **04** | REVEL, PrimateAI — supervised / semi-supervised (+circularity) | demo |
| **05** | ClinVar, CFTR2 — the clinical & functional truth sets | ClinVar REAL |
| **06** | SpliceAI, Pangolin, CADD — splicing | CADD REAL |
| **07** | **Integration** — reproduce the A1/A2 worklists honestly | mixed |
| **08** | **Circular reasoning & training leakage** — the methodology | — |

Recommended order: 01 → 08 in sequence. If you only read two, read **07** (what
the numbers really are) and **08** (why to be careful).

## To swap in REAL data for the demo tools

Each tool notebook has a *"How to get the REAL data"* section. In short:

- **EVE** — per-protein CSV from evemodel.org (CFTR = UniProt P13569)
- **ESM1b** — bulk variant-effect files (Brandes 2023 supplement / HuggingFace)
- **REVEL** — genome-wide table from sites.google.com/site/revelgenomics
- **PrimateAI** — Illumina distribution
- **SpliceAI** — precomputed VCF (Illumina BaseSpace) or the Broad lookup app
- **Pangolin** — run locally (needs GPU)
- **CFTR2** — cftr2.org (accept data-use terms)

Join each on `protein_variant` (missense) or genomic `chr/pos/ref/alt` (splice),
set the loader's `source` to `"REAL"`, and re-run notebook 07.